# NB12 — Q4 — Is this a new idea, or an old one renamed?

        **CPU only — turn the accelerator OFF. ~20 minutes. One account.**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## The question

        > Is 'how much compute does this image need' just 'how hard is this image', which people already measure?

        ## In plain English

        We fit a predictor using seven existing difficulty scores (model confidence, entropy, loss, how often the model forgot the image during training, and so on). Then we add our measurement and see whether the prediction improves. If it doesn't improve, our measurement carries no new information.

        ## Why it matters

        **This is the biggest threat to the project and we test it head-on rather than hoping nobody asks.** If it turns out MSC is fully explained by existing scores, that is still publishable and we must not hide it: 'per-sample compute requirements are fully explained by classical difficulty scores' saves the community effort, and the engineering result that follows — use a cheap score instead of an expensive oracle — is arguably more useful than the method paper.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   21b2b87a46a2   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2571e11b4f7e   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Load

In [ ]:
ACCOUNT = 'acct1'      # <<< CHANGE ME

sess = msc.Session(account=ACCOUNT, phase='analysis', dataset='cifar100',
                   enable_hf=True)
# Metrics and per-sample tables only -- checkpoints excluded. Fast.
sess.sync_state(include_checkpoints=False, verbose=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt

# Inventory: what do we actually have to work with?
runs = {}
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    ps = d / 'per_sample'
    m = msc.read_json(ps / 'meta.json', default=None)
    if m and (ps / 'test.parquet').exists():
        runs[d.name] = m
budgets = {r: sess.budgets(m['arch']) for r, m in runs.items()}

inv = pd.DataFrame([{'run_id': k, 'arch': v['arch'], 'family': v['family'],
                     'seed': v['seed'],
                     'order': v['sample_order_hash'][:10]} for k, v in runs.items()])
if len(inv):
    inv = inv.sort_values(['family', 'arch', 'seed'])
    display(inv)
    print(f"\n{len(runs)} measured models   "
          f"{inv.order.nunique()} distinct image orderings (must be 1)")
else:
    # Not an error -- it means the measurement notebook has not run yet on any
    # model this account can see.
    trained = [d.name for d in sorted(sess.runs_dir.iterdir())
               if (d / 'summary.json').exists()] if sess.runs_dir.exists() else []
    print('No measured models found.')
    print()
    if trained:
        print(f'{len(trained)} run(s) have finished TRAINING but not MEASUREMENT:')
        for t in trained[:10]:
            print(f'  {t}')
        print()
        print('-> Run NB02 (Phase 0) or NB08 (atlas) to produce the per-sample')
        print('   tables, then re-run this notebook.')
    else:
        print('-> No completed runs at all. Run sess.sync_state(), or finish')
        print('   the training notebooks first.')

## Step 2 — Is the difficulty battery complete?

Seven scores. Three of them (EL2N, forgetting events, prediction depth)
can only be recorded *during* training — they can't be recovered from a
finished model. If they're missing, this analysis is weaker than it
should be and NB08 needs re-running.

In [ ]:
battery = ('msp', 'margin', 'entropy', 'ce_loss', 'el2n', 'forget_events',
           'pred_depth')
rows = []
for r in list(runs)[:6]:
    try:
        df = msc.load_per_sample(sess.data_dir, r, 'train_holdout')
    except FileNotFoundError:
        continue
    rows.append({'run_id': r,
                 **{c: (int(df[c].notna().sum()) if c in df.columns else 0)
                    for c in battery}})
cov = pd.DataFrame(rows)
if not len(cov):
    print('No per-sample tables found. Run NB08 (or NB02 for Phase 0) first.')
    missing = list(battery)
else:
    display(cov)
    missing = [c for c in battery if cov[c].sum() == 0]
if missing:
    print(f'\nMISSING: {missing}')
    print('Q4 will be weaker. These come from training-time instrumentation;')
    print('re-run NB08 with train_dynamics present to recover them.')
else:
    print('\nAll seven difficulty scores present.')

## Step 3 — The test

Two things reported:

- **partial correlation** — do two models still agree about compute-need
  *after* removing everything the difficulty scores explain?
- **ΔR²** — how much better can we predict one model's compute-need when
  we add another model's, on top of the difficulty scores?

Prediction (H4): ΔR² ≥ 0.05 and partial correlation ≥ 0.3.

In [ ]:
import itertools
ceilings = msc.read_json(sess.data_dir / 'analysis' / 'ceilings.json', default={}) or {}
seed1 = {m['arch']: r for r, m in runs.items() if m['seed'] == 1}
pairs = list(itertools.combinations(sorted(seed1), 2))[:15]

q4_all = []
for a, b in pairs:
    try:
        d = msc.analyse_q4_irreducibility(sess.data_dir, seed1[a], seed1[b],
                                          budgets, axis='depth', taus=(0.1,),
                                          n_boot=500)
        d['arch_a'] = a; d['arch_b'] = b
        q4_all.append(d)
    except Exception as e:
        print(f'  {a} -> {b}: {e}')
q4 = pd.concat(q4_all, ignore_index=True) if q4_all else pd.DataFrame()
if not len(q4):
    print('No pairs analysed. Q4 needs at least two measured architectures.')
if len(q4):
    msc.save_analysis(sess.data_dir, 'q4_irreducibility_all', q4, sess.hub)
    display(q4[['arch_a', 'arch_b', 'partial_spearman', 'r2_difficulty_only',
                'r2_difficulty_plus_msc', 'delta_r2', 'delta_r2_lo',
                'delta_r2_hi']].round(4))
    print(f"\n  median delta R2       = {q4.delta_r2.median():.4f}   (H4 wants >= 0.05)")
    print(f"  median partial corr   = {q4.partial_spearman.median():.4f}   (H4 wants >= 0.30)")
    frac = (q4.delta_r2 >= 0.05).mean()
    print(f"  pairs clearing H4     = {frac:.0%}")

## Step 4 — Plot

In [ ]:
if len(q4):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    lbl = q4.arch_a + '->' + q4.arch_b
    ax[0].barh(lbl, q4.delta_r2,
               xerr=[q4.delta_r2 - q4.delta_r2_lo, q4.delta_r2_hi - q4.delta_r2])
    ax[0].axvline(0.05, ls='--', c='g', lw=1, label='H4 pass')
    ax[0].axvline(0.02, ls=':', c='r', lw=1, label='reframe below this')
    ax[0].set_xlabel('delta R^2 (information beyond difficulty scores)')
    ax[0].legend(); ax[0].grid(alpha=.3)
    ax[0].tick_params(labelsize=7)

    ax[1].scatter(q4.r2_difficulty_only, q4.r2_difficulty_plus_msc)
    lo = float(min(q4.r2_difficulty_only.min(), q4.r2_difficulty_plus_msc.min()))
    hi = float(max(q4.r2_difficulty_only.max(), q4.r2_difficulty_plus_msc.max()))
    ax[1].plot([lo, hi], [lo, hi], 'k--', lw=1, label='no improvement')
    ax[1].set_xlabel('R^2 using difficulty scores alone')
    ax[1].set_ylabel('R^2 adding our measurement')
    ax[1].set_title('Points above the line = we add information')
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout()
    msc.save_figure(fig, sess.data_dir, 'q4_irreducibility', sess.hub)
    plt.show()

## Step 5 — Read the verdict honestly

Whichever way this comes out, it determines the shape of the paper.

In [ ]:
if len(q4):
    med = float(q4.delta_r2.median())
    if med >= 0.05:
        print('VERDICT: MSC carries information beyond classical difficulty scores.')
        print('  -> The construct is new. Keep the multi-axis oracle. Full paper.')
    elif med >= 0.02:
        print('VERDICT: Marginal. Some information beyond difficulty, not much.')
        print('  -> Report honestly, temper the novelty claim.')
    else:
        print('VERDICT: MSC is largely explained by existing difficulty scores.')
        print('  -> REFRAME. This is a real, useful, citable finding: cheap scores')
        print('     suffice for compute routing. Do NOT hide it. The method section')
        print('     becomes "use a difficulty score", which is a BETTER engineering')
        print('     result than an expensive multi-axis oracle.')

## Step 6 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()